# ShopSphere E7: tuned GRU + MLP with profile and regional features (GPU)

**E7 = E4T's tuned settings + 12 extra static columns** (`src/shopsphere_dl/features_extra.py`):
- 8 profile and consent columns (`loyalty_score`, `email_open_rate`, `avg_review_score`, opt-ins,
  `preferred_channel`...), filled in on the **30 Sep 2024 snapshot only** and blank on the 12 earlier
  snapshots (enforced by `tests/test_extra_features.py`);
- 4 regional columns from `geo_data`, on every snapshot.

Inputs: **`shopsphere-data`** (`shopsphere.db`, `folds.csv`) and **`shopsphere-code`** (new version
with `features_extra.py`; its `oof/` folder holds E4T, `02`, `03` and the 80/20 holdout files).

Settings: **GPU T4**, then **Save Version → Save & Run All**.

**Plan:** run `SEEDS = [1]` first (5 folds, about an hour). If E7 beats E4T, save a new version with
`SEEDS = [2, 3, 4, 5]` and this version's output attached as an input: its `oof_E7_seed1.csv` and
checkpoints are picked up automatically.

In [ ]:
import glob, os, sys, subprocess, shutil, zipfile
import pandas as pd

def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    if not hits:
        for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):      # in case Kaggle kept a zip
            zipfile.ZipFile(z).extractall('/kaggle/working/unzipped')
        hits = sorted(glob.glob(f'/kaggle/working/unzipped/**/{pattern}', recursive=True))
    assert hits, f'{pattern} not found: attach the datasets to this notebook'
    return hits

DATA = os.path.dirname(find('shopsphere.db')[0])
CODE = os.path.dirname(os.path.dirname(os.path.dirname(find('shopsphere_dl/features_extra.py')[0])))
print('data:', DATA); print('code:', CODE)
assert os.path.exists(os.path.join(CODE, 'best_params.json')), 'best_params.json missing from the code dataset'

os.environ.update(SHOPSPHERE_DB=os.path.join(DATA, 'shopsphere.db'),
                  SHOPSPHERE_FOLDS=os.path.join(DATA, 'folds.csv'),
                  SHOPSPHERE_OUT='/kaggle/working',
                  PYTHONPATH=os.path.join(CODE, 'src'))
sys.path.insert(0, os.path.join(CODE, 'src'))

import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (turn on GPU T4)')

# Resume / add seeds: files from an earlier version attached as an input are copied here.
for pattern in ['ckpt_E7_*.pt', 'oof_E7_seed*.csv', 'history_E7_*.json']:
    for f in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True):
        dst = os.path.join('/kaggle/working', os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst); print('from earlier version:', os.path.basename(f))

## 1. Tests: leakage, model, extra features, fairness. Training starts only if all pass.

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', os.path.join(CODE, 'tests')],
                   capture_output=True, text=True, cwd='/kaggle/working')
print(r.stdout[-3000:], r.stderr[-1500:])
assert r.returncode == 0, 'tests failed: do not train until they pass'

## 2. Train E7

Same tuned settings as E4T (`best_params.json`), no re-tuning. One line per epoch: training loss,
validation PR-AUC and seconds. A seed whose `oof_E7_seed<s>.csv` already exists is skipped.

In [ ]:
SEEDS = [1]                      # first run; then [2, 3, 4, 5]

for seed in SEEDS:
    if os.path.exists(f'/kaggle/working/oof_E7_seed{seed}.csv'):
        print(f'seed {seed}: already done'); continue
    cmd = [sys.executable, '-m', 'shopsphere_dl.train', '--run', 'E4', '--seed', str(seed),
           '--params', os.path.join(CODE, 'best_params.json'), '--name', 'E7', '--extra', 'crm,geo']
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd='/kaggle/working')
    for line in p.stdout:
        if 'Warning' not in line:
            print(line, end='')
    assert p.wait() == 0, f'E7 seed {seed} failed'

## 3. Compare with E4T and the baselines, and re-check bias

- 5-fold on all 64,000 customers, and the team's 80/20 holdout (12,800 customers) with the team's rule
  (ROC-AUC change + PR-AUC change over `02`), against both the team-reported `02` and our rerun.
- Paired bootstrap of E7 against E4T **with the same number of seeds**.
- Bias: calibration and recall by age band and region tier, then the age allocation.

In [ ]:
r = subprocess.run([sys.executable, os.path.join(CODE, 'scripts', 'compare_e7.py'),
                    '--new-dir', '/kaggle/working', '--ref-dir', os.path.join(CODE, 'oof'), '--out', '/kaggle/working'],
                   capture_output=True, text=True, cwd='/kaggle/working')
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0

Download from the Output tab: `oof_E7_seed*.csv`, `compare_5fold.csv`, `compare_holdout.csv`,
`bootstrap_e7.csv`, `bias_e7.csv` and the `history_E7_*.json` files. Keep the `ckpt_E7_*.pt` files
in the output: the export step needs them.